In [ ]:
pred_tile_dir = "Tiles"
index_csv_path = os.path.join("Tiles", "tile_index.csv")

In [ ]:
from torch.utils.data import DataLoader

class PredictionJPGDataset(Dataset):
    def __init__(self, img_dir, index_csv, transforms=None):
        self.img_dir = img_dir
        self.df = pd.read_csv(index_csv)
        self.image_list = self.df['tile'].tolist()
        self.transforms = transforms

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        # Change the extension from the CSV list to .jpg
        fname = self.image_list[idx].replace('.tif', '.jpg')
        img_path = os.path.join(self.img_dir, fname)

        # Check if the file exists and is the right size
        if not os.path.exists(img_path):
            return self.__getitem__((idx + 1) % len(self.image_list))

        img = Image.open(img_path).convert("RGB")

        # Skip edge tiles (not 1024x1024)
        if img.size != (1024, 1024):
            return self.__getitem__((idx + 1) % len(self.image_list))

        if self.transforms:
            img = self.transforms(img)

        return img, fname

#Standard ToTensor: scales 0-255 to 0.0-1.0
predict_dataset = PredictionJPGDataset(pred_tile_dir, index_csv_path, transforms=transforms.ToTensor())
# Re-initialize the loader
predict_loader = DataLoader(predict_dataset, batch_size=4, shuffle=False)

print(f"Total tiles in index: {len(predict_dataset)}")
print("Note: Partial edge tiles will be skipped automatically during iteration.")

In [ ]:
model.eval()
all_predictions = {}

print("Running predictions on JPGs...")

with torch.no_grad():
    for imgs, fnames in predict_loader:
        imgs = imgs.to(device)
        preds = model(imgs)

        # The decode_centers function expects raw logits and handles sigmoid internally.
        # So we pass the original 'preds' which contains heatmap logits and offsets.
        decoded_list = decode_centers(
            preds,
            stride=8,
            conf_thresh=0.4,
            min_dist=100
        )

        for i, detections in enumerate(decoded_list):
            all_predictions[fnames[i]] = detections

print(f"Done! Processed {len(all_predictions)} tiles.")

In [ ]:
all_points = []

print("Converting pixel predictions to geographic coordinates...")

for tif_name, points in all_predictions.items():
    # Map the JPG name back to the original TIF path
    original_tif = os.path.join(pred_tile_dir, tif_name.replace('.jpg', '.tif'))

    if not os.path.exists(original_tif):
        continue

    with rasterio.open(original_tif) as src:
        tile_transform = src.transform
        tile_crs = src.crs

        for p in points:
            # Convert GPU tensor elements to Python floats
            px, py = p[0].cpu().item(), p[1].cpu().item()

            # rasterio transforms map coordinates
            # ~src.transform maps (lon, lat) -> (px, py)
            # src.transform maps (px, py) -> (lon, lat)
            gx, gy = tile_transform * (px, py)

            all_points.append({
                'geometry': Point(gx, gy),
                'tile': tif_name,
                'confidence': 1.0 # As decode_centers returns only px, py
            })

# Create a GeoDataFrame
if all_points:
    gdf = gpd.GeoDataFrame(all_points, crs=tile_crs)

    # Save to Google Drive
    output_shp = os.path.join("Data", "all_seedling_predictions.shp")
    gdf.to_file(output_shp)
    print(f"Success! Saved {len(gdf)} seedlings to {output_shp}")
else:
    print("No points found to save.")